# Введение в MapReduce модель на Python


In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(0.5813610518995423)),
 (1, np.float64(0.5813610518995423)),
 (2, np.float64(0.5813610518995423)),
 (3, np.float64(0.5813610518995423)),
 (4, np.float64(0.5813610518995423))]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('it', 18), ('what', 10)]),
 (1, [('a', 2), ('is', 18)])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.003483024213998931)),
   (None, np.float64(0.025563815393070843)),
   (None, np.float64(0.05406660058977075)),
   (None, np.float64(0.1079221665898007)),
   (None, np.float64(0.13062164519026287)),
   (None, np.float64(0.13184171773403863)),
   (None, np.float64(0.1329888139340295)),
   (None, np.float64(0.16519043568998049)),
   (None, np.float64(0.19520213351757554)),
   (None, np.float64(0.24452757296355843)),
   (None, np.float64(0.3115478895627838)),
   (None, np.float64(0.3151346528084582)),
   (None, np.float64(0.3769214559586045)),
   (None, np.float64(0.43644711345059495)),
   (None, np.float64(0.49518880002341936))]),
 (1,
  [(None, np.float64(0.5816519175709067)),
   (None, np.float64(0.5841392056290523)),
   (None, np.float64(0.6207184046522056)),
   (None, np.float64(0.6450318685651797)),
   (None, np.float64(0.6757083698383464)),
   (None, np.float64(0.6774751742684438)),
   (None, np.float64(0.6833794648140448)),
   (None, np.float64(0.7828131

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
numbers = [1, 4, 2, 9, 7, 8, 5]

def RECORDREADER():
    return [(None, x) for x in numbers]

def MAP(_, value):
    yield (1, value)

def REDUCE(key, values):
    yield (None, max(values))

out = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Максимальное значение ряда =", out[0][1])

Максимальное значение ряда = 9


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
def MAP(_, value):
    yield (1, (value, 1))

def REDUCE(key, values):
    total_sum = 0
    count = 0
    for v, c in values:
        total_sum += v
        count += c
    yield (None, total_sum / count)

out = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Арифметическое среднее =", out[0][1])

Арифметическое среднее = 5.142857142857143


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
def groupbykey_sort(iterable):
    # cортировка по ключу k2
    sorted_iterable = sorted(iterable, key=lambda x: x[0])

    if not sorted_iterable:
        return []

    # группировка последовательных ключей
    result = []
    current_key, current_values = sorted_iterable[0][0], [sorted_iterable[0][1]]

    for k, v in sorted_iterable[1:]:
        if k == current_key:
            current_values.append(v)
        else:
            result.append((current_key, current_values))
            current_key, current_values = k, [v]
    result.append((current_key, current_values))
    return result

# проверка
test = [(2, 'a'), (1, 'b'), (2, 'c')]
print("GroupByKey на основе сортировки:", list(groupbykey_sort(test)))

GroupByKey на основе сортировки: [(1, ['b']), (2, ['a', 'c'])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
data = [1, 2, 12, 3, 4, 4, 4, 5, 12, 2]

def RECORDREADER():
    return [(i, x) for i, x in enumerate(data)]

def MAP(_, x):
    yield (x, None)

def REDUCE(x, _):
    yield x

elements = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Исключение дубликатов:", elements)

Исключение дубликатов: [1, 2, 12, 3, 4, 5]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
def MAP_SELECT(_, row):
    if row.age > 30:
        yield (row, row)

def REDUCE_IDENTITY(key, values):
    yield key

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
def MAP_PROJECT(_, row):
    t_prime = (row.id, row.gender)
    yield (t_prime, t_prime)

def REDUCE_PROJECT(key, values):
    yield key

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
def MAP_UNION(_, row):
    yield (row, row)

def REDUCE_UNION(key, values):
    yield key

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
def REDUCE_INTERSECT(key, values):
    if len(list(values)) >= 2:
        yield key

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
def MAP_DIFF(origin, row):
    yield (row, origin)

def REDUCE_DIFF(key, origins):
    hist = list(origins)
    if 'R' in hist and 'S' not in hist:
        yield key

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
def MAP_JOIN(rel_name, row):
    if rel_name == 'R':
        a, b = row
        yield (b, ('R', a))
    else:
        b, c = row
        yield (b, ('S', c))

def REDUCE_JOIN(b, values):
    list_values = list(values)
    r_parts = [v[1] for v in list_values if v[0] == 'R']
    s_parts = [v[1] for v in list_values if v[0] == 'S']
    for a in r_parts:
        for c in s_parts:
            yield (a, b, c)

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
data = [('A', 10, 1), ('B', 5, 2), ('A', 20, 3)]

def RECORDREADER():
    return [(None, row) for row in data]

def MAP(_, row):
    yield (row[0], row[1])

def REDUCE(key, values):
    yield (key, sum(values))

list(MapReduce(RECORDREADER, MAP, REDUCE))

#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
def MAP(k1, v1):
    (j, k) = k1
    w = v1
    for i in range(small_mat.shape[0]):
        v = small_mat[i, j]
        yield ((i, k), v * w)

def REDUCE(key, values):
    (i, k) = key
    yield (key, sum(values))

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [ ]:
import numpy as np
I = 2
J = 3
K = 4*10

small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  # solution code that yield(k2,v2) pairs
  for i in range(small_mat.shape[0]):
        v = small_mat[i, j]
        yield ((i, k), v * w)

def REDUCE(key, values):
  (i, k) = key
  # solution code that yield(k3,v3) pairs
  yield ((i, k), sum(values))

Проверьте своё решение

In [ ]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)

  if not reduce_output:
        raise ValueError("MapReduce вернул пустой результат")

  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.zeros((int(I), int(K)))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

solution_matrix = asmatrix(solution)

# should return true
np.allclose(reference_solution, solution_matrix)

True

In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [ ]:
I, J, K = 2, 3, 2
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

def RECORDREADER():
    for i, j in np.ndindex(mat_M.shape): yield (None, ('M', i, j, mat_M[i,j]))
    for j, k in np.ndindex(mat_N.shape): yield (None, ('N', j, k, mat_N[j,k]))

def MAP(_, data):
    tag, r, c, val = data
    if tag == 'M':
        for k in range(K): yield ((r, k), (c, val))
    else:
        for i in range(I): yield ((i, c), (r, val))

def REDUCE(key, values):
  res = {}
  for j, val in values:
      res[j] = res.get(j, 1) * val

  from collections import defaultdict
  d = defaultdict(list)
  for j, val in values: d[j].append(val)
  yield (key, sum(v[0] * v[1] for v in d.values() if len(v) == 2))

solution = MapReduce(RECORDREADER, MAP, REDUCE)
print("Результат:", np.allclose(np.matmul(mat_M, mat_N), asmatrix(solution)))

Результат: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [ ]:
maps = 2
reducers = 2

def INPUTFORMAT():
    def RR_M():
        for i, j in np.ndindex(mat_M.shape):
            yield (None, ('M', i, j, mat_M[i, j]))

    def RR_N():
        for j, k in np.ndindex(mat_N.shape):
            yield (None, ('N', j, k, mat_N[j, k]))

    yield RR_M()
    yield RR_N()

def MAP(_, data):
    tag, r, c, val = data
    if tag == 'M':
        for k in range(K):
            yield ((r, k), ('M', c, val))
    else:
        for i in range(I):
            yield ((i, c), ('N', r, val))

def REDUCE(key, values):
    m_vals = {}
    n_vals = {}
    for tag, j, val in values:
        if tag == 'M': m_vals[j] = val
        else: n_vals[j] = val

    res = sum(m_vals[j] * n_vals[j] for j in m_vals if j in n_vals)
    yield (key, res)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

flat_output = []
for part_id, part_gen in partitioned_output:
    flat_output.extend(list(part_gen))

print("Результат:", np.allclose(np.matmul(mat_M, mat_N), asmatrix(flat_output)))

24 key-value pairs were sent over a network.
Результат: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

Ответ: Будет

In [ ]:
I, J, K = 4, 4, 4
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

maps_per_matrix = 3
reducers = 4

def get_elements(matrix, tag):
    elements = []
    for r, c in np.ndindex(matrix.shape):
        elements.append((tag, r, c, matrix[r, c]))
    np.random.shuffle(elements)
    return elements

elements_M = get_elements(mat_M, 'M')
elements_N = get_elements(mat_N, 'N')

def INPUTFORMAT():
    def create_reader(chunk):
        def RECORDREADER():
            for item in chunk:
                yield (None, item)
        return RECORDREADER()

    size_m = int(np.ceil(len(elements_M) / maps_per_matrix))
    for i in range(0, len(elements_M), size_m):
        yield create_reader(elements_M[i:i+size_m])

    size_n = int(np.ceil(len(elements_N) / maps_per_matrix))
    for i in range(0, len(elements_N), size_n):
        yield create_reader(elements_N[i:i+size_n])

def MAP(_, data):
    tag, r, c, val = data
    if tag == 'M':
        for k in range(K):
            yield ((r, k), ('M', c, val))
    else:
        for i in range(I):
            yield ((i, c), ('N', r, val))

def REDUCE(key, values):
    m_dict = {}
    n_dict = {}
    for tag, j, val in values:
        if tag == 'M': m_dict[j] = val
        else: n_dict[j] = val

    res = sum(m_dict[j] * n_dict[j] for j in m_dict if j in n_dict)
    yield (key, res)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

flat_output = []
for part_id, part_gen in partitioned_output:
    flat_output.extend(list(part_gen))

print("Результат:", np.allclose(np.matmul(mat_M, mat_N), asmatrix(flat_output)))

128 key-value pairs were sent over a network.
Результат: True
